In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
ResultsPaths = object


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- eval_hybrid_concat_sort ---
FIX_EVAL_HYBRID_CONCAT_SORT_FRAMES_BEFORE = [
    pd.DataFrame({"Query Document ID":[101,102],"Best Score":[0.9,0.7]}),
    pd.DataFrame({"Query Document ID":[103,104],"Best Score":[0.6,0.5]}),
]
FIX_EVAL_HYBRID_CONCAT_SORT_FRAMES_GEN = [
    pl.DataFrame({"Query Document ID":[101,102],"Best Score":[0.9,0.7]}),
    pl.DataFrame({"Query Document ID":[103,104],"Best Score":[0.6,0.5]}),
]

# --- eval_hybrid_load ---
FIX_EVAL_HYBRID_LOAD_LOAD_DF_FROM_PICKLE = lambda path: pd.DataFrame({"id":[1,2,3],"title":["A","B","C"],"label":[0,1,0]})
FIX_EVAL_HYBRID_LOAD_TFIDF_COSINE_SIMILARITIES_MOST_CITED = pd.DataFrame({"id":[0,1,2],"score":[0.9,0.7,0.5]}).set_index("id")
ResultsPaths = SimpleNamespace(language_models=SimpleNamespace(tfidf_cosine_similarities_most_cited_pkl="tfidf_cosine_similarities_most_cited.pkl"))

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_eval_hybrid_concat_sort(frames):
    average_precision_scores = pd.concat(frames)
    return (
    average_precision_scores
    .sort_values("Best Score", ascending=False)
    .reset_index(drop=False)
    .drop_duplicates(subset="Query Document ID")
    .reset_index(drop=True)
    )
    return average_precision_scores

def before_eval_hybrid_load(load_df_from_pickle, tfidf_cosine_similarities_most_cited):
    tfidf_cosine_similarities_most_cited: pd.DataFrame = load_df_from_pickle(
        ResultsPaths.language_models.tfidf_cosine_similarities_most_cited_pkl
    )
    return None

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_eval_hybrid_concat_sort(frames):

    average_precision_scores = pl.concat(frames)
    return (
        average_precision_scores
        .sort("Best Score", descending=True)
        .unique(subset="Query Document ID", keep="first", maintain_order=True)
    )

def gen_eval_hybrid_load(load_df_from_pickle, tfidf_cosine_similarities_most_cited):

    tfidf_cosine_similarities_most_cited: pl.DataFrame = load_df_from_pickle(
        ResultsPaths.language_models.tfidf_cosine_similarities_most_cited_pkl
    )
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: eval_hybrid_concat_sort ===

# L1 smoke – generated
try:
    _r = gen_eval_hybrid_concat_sort(FIX_EVAL_HYBRID_CONCAT_SORT_FRAMES_GEN)
    print("✅ L1 smoke gen_eval_hybrid_concat_sort: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_eval_hybrid_concat_sort: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_eval_hybrid_concat_sort(FIX_EVAL_HYBRID_CONCAT_SORT_FRAMES_BEFORE)
    print("✅ L1 smoke before_eval_hybrid_concat_sort: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_eval_hybrid_concat_sort: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_eval_hybrid_concat_sort(FIX_EVAL_HYBRID_CONCAT_SORT_FRAMES_BEFORE)
    _rg = gen_eval_hybrid_concat_sort(FIX_EVAL_HYBRID_CONCAT_SORT_FRAMES_GEN)
    compare(_rb, _rg, "eval_hybrid_concat_sort", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence eval_hybrid_concat_sort: setup error — {type(_e).__name__}: {_e}")

# AUDIT-75: duplicate IDs exercise score ordering and keep-first semantics.
try:
    _frames_pd = [
        pd.DataFrame({"Query Document ID": [101, 102], "Best Score": [0.4, 0.8]}),
        pd.DataFrame({"Query Document ID": [101, 103], "Best Score": [0.9, 0.3]}),
    ]
    _frames_pl = [pl.from_pandas(frame) for frame in _frames_pd]
    _rb = before_eval_hybrid_concat_sort(_frames_pd)
    _rg = gen_eval_hybrid_concat_sort(_frames_pl)
    compare(_rb, _rg, "L3 edge eval_hybrid_concat_sort duplicate IDs", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge eval_hybrid_concat_sort duplicate IDs: {type(_e).__name__}: {_e}")
